In [ ]:
# Image-Only Inference
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
from pathlib import Path

# Get inference device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model - This requires internet access or the huggingface hub cache to be pre-downloaded
# For Apache 2.0 license model, use "facebook/map-anything-apache"
model = MapAnything.from_pretrained("facebook/map-anything-apache").to(device)

# Load and preprocess images from a folder or list of paths
images = ["/workspace/data/nuscenes/samples/CAM_FRONT/n008-2018-05-21-11-06-59-0400__CAM_FRONT__1526915250362465.jpg",
          "/workspace/data/nuscenes/samples/CAM_FRONT/n008-2018-05-21-11-06-59-0400__CAM_FRONT__1526915250862465.jpg",
          "/workspace/data/nuscenes/samples/CAM_FRONT/n008-2018-05-21-11-06-59-0400__CAM_FRONT__1526915251362465.jpg"]  # or ["path/to/img1.jpg", "path/to/img2.jpg", ...]
views = load_images(images)

# Run inference
predictions = model.infer(
    views,                            # Input views
    memory_efficient_inference=True,  # Trades off speed for more views (up to 2000 views on 140 GB). Trade off is negligible - see profiling section
    minibatch_size=None,              # Minibatch size for memory-efficient inference (use 1 for smallest GPU memory consumption). Default is dynamic computation based on available GPU memory.
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
    use_multiview_confidence=False,   # Enable multi-view depth consistency based confidence in place of learning-based one
)

# Access results for each view - Complete list of metric outputs
for i, pred in enumerate(predictions):
    # Geometry outputs
    pts3d = pred["pts3d"]                     # 3D points in world coordinates (B, H, W, 3)
    pts3d_cam = pred["pts3d_cam"]             # 3D points in camera coordinates (B, H, W, 3)
    depth_z = pred["depth_z"]                 # Z-depth in camera frame (B, H, W, 1)
    depth_along_ray = pred["depth_along_ray"] # Depth along ray in camera frame (B, H, W, 1)

    # Camera outputs
    ray_directions = pred["ray_directions"]   # Ray directions in camera frame (B, H, W, 3)
    intrinsics = pred["intrinsics"]           # Recovered pinhole camera intrinsics (B, 3, 3)
    camera_poses = pred["camera_poses"]       # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world poses in world frame (B, 4, 4)
    cam_trans = pred["cam_trans"]             # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world translation in world frame (B, 3)
    cam_quats = pred["cam_quats"]             # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world quaternion in world frame (B, 4)

    # Quality and masking
    confidence = pred["conf"]                 # Per-pixel confidence scores (B, H, W)
    mask = pred["mask"]                       # Combined validity mask (B, H, W, 1)
    non_ambiguous_mask = pred["non_ambiguous_mask"]                # Non-ambiguous regions (B, H, W)
    non_ambiguous_mask_logits = pred["non_ambiguous_mask_logits"]  # Mask logits (B, H, W)

    # Scaling
    metric_scaling_factor = pred["metric_scaling_factor"]  # Applied metric scaling (B,)

    # Original input
    img_no_norm = pred["img_no_norm"]         # Denormalized input images for visualization (B, H, W, 3)

In [ ]:
# Visualize the input images and the predicted depth maps (depth_z is metric depth in meters)
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(predictions), 2, figsize=(14, 4 * len(predictions)), squeeze=False)
for i, pred in enumerate(predictions):
    image = pred["img_no_norm"][0].cpu().numpy()            # (H, W, 3)
    depth = pred["depth_z"][0, ..., 0].float().cpu().numpy()  # (H, W) [m]
    valid = pred["mask"][0, ..., 0].cpu().numpy().astype(bool)
    depth_masked = np.where(valid, depth, np.nan)  # invalid pixels are shown as blank

    axes[i, 0].imshow(image)
    axes[i, 0].set_title(f"View {i}: input image")
    axes[i, 0].axis("off")
    im = axes[i, 1].imshow(depth_masked, cmap="turbo",
                           vmin=np.nanpercentile(depth_masked, 2), vmax=np.nanpercentile(depth_masked, 98))
    axes[i, 1].set_title(f"View {i}: depth_z [m] (scale factor {pred['metric_scaling_factor'][0].item():.2f})")
    axes[i, 1].axis("off")
    fig.colorbar(im, ax=axes[i, 1], fraction=0.03, label="depth [m]")
    print(f"View {i}: depth range {np.nanmin(depth_masked):.1f} - {np.nanmax(depth_masked):.1f} m, median {np.nanmedian(depth_masked):.1f} m")
plt.tight_layout()
plt.show()